# Project 5: Supervised fine-tuning for candle-data repair instructions

**Name:** To be completed  
**Course:** CSCI 4820 - section to be completed  
**Project:** 5

The model receives corrupted candle CSV and predicts a JSON tool-use plan:
`{"recoverable": true, "operations": [...]}` or
`{"recoverable": false, "operations": []}`. A negative decision omits the complete
window during final cleaning; negative examples remain in supervised training.

The CPU oracle deterministically verifies labels against privileged originals.
The model never receives the original data or corruption history. Run the cells
in order: inspect data, construct SFT examples, observe the untrained model, train
on all 10,000 examples, then compare the trained model on the same inputs.
Full model stages are explicit notebook cells and may take several hours.

**Execution scope:** the main path is three before-SFT predictions, one full Trainer-based SFT run using offline labels, and five after-SFT predictions (three matched and two additional). Full-set generation is an explicit optional evaluation job below, not dataset construction.


## 1. Environment and SFT configuration

Install the published CPU library once in the active notebook environment with
`%pip install candleprep==0.1.0`. Obtain the source CSV separately from the project
repository. The existing Mistral/Transformers/PEFT dependencies are still required.
Locally, use the global Python 3.11.9 CUDA kernel; final execution must also be
verified on TAMU Launch's 24 GB A30.

The main workflow trains one rank-16 adapter on all 10,000 cached labeled pairs
for one epoch, like the supplied starter's Trainer-based LoRA approach. Change
`LORA_RANK` to configure that run. Rank studies and full test generation are
separate, disabled-by-default evaluation sections, so they do not block SFT.
Editing a switch does not change a running kernel. Rerun the configuration
cell, then explicitly run the desired Section 12 cell; neither job starts
automatically. The configuration output shows the active switch values.

The completed rank-16 run peaked at 5.90 GiB allocated by PyTorch; its
live kernel retained a 6.17 GiB peak reservation after training/inference.
These exclude memory outside the PyTorch allocator; whole-device peak
was not recorded. A 10-GB GPU is a reasonable next test with the same
batch/sequence settings, but still needs verification on that GPU.
`MIN_FREE_VRAM_GIB` replaces the old 12-GiB free-memory guard. Future
training reports save both allocated and reserved peaks. See
[PyTorch memory accounting](https://docs.pytorch.org/docs/2.10/notes/cuda.html#memory-management).

After interrupting a run, wait until the kernel is idle, then execute
`free_gpu_resources()` in a new cell. This removes live model/trainer
references, traceback locals, and cached Python outputs before collecting
garbage and releasing unused CUDA memory. Dataset variables and saved
checkpoints remain available. Rerun the interrupted execution cell to
reload its model; training can resume from its latest saved checkpoint.
If CUDA still reports an error, restart the kernel.


In [ ]:
import gc
import hashlib
import json
import math
import platform
from pathlib import Path
from collections import Counter


SOURCE = Path("TVC_GOLD, 360.csv")
DATA_DIR = Path("generated_data/v1")
RESULTS = Path("candle_results/full_sft_v1")
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
SEED = 4820
LORA_RANK = 16
RANKS = [8, 16, 32]  # Optional rank study only
NUM_EPOCHS = 1
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 16
LEARNING_RATE = 2e-4
MAX_LENGTH = 1536
MAX_NEW_TOKENS = 128
MIN_FREE_VRAM_GIB = 7.5  # Availability guard, not a guarantee of fit on another GPU
RUN_RANK_STUDY = True
RUN_FULL_TEST_COMPARISON = True
RUN_FINAL_DEMO = False
RESULTS.mkdir(parents=True, exist_ok=True)
print("Full SFT configuration:", {"primary_rank": LORA_RANK, "epochs_per_rank": NUM_EPOCHS,
      "batch_size": BATCH_SIZE, "gradient_accumulation": GRADIENT_ACCUMULATION})

print("Active optional jobs:", {"RUN_RANK_STUDY": RUN_RANK_STUDY,
      "RUN_FULL_TEST_COMPARISON": RUN_FULL_TEST_COMPARISON})


### Manual cleanup after interruption

Run this definition cell once. After an interrupted run stops, call
`free_gpu_resources()` in a new cell. After a full kernel restart,
restore setup using the checklist in Section 11.


In [ ]:
def free_gpu_resources():
    """Call in a new cell after an interrupted run has stopped."""
    import sys
    import traceback
    import torch

    # Drop this notebook's live models/training objects; keep data and checkpoints.
    for name in ("trainer", "model", "base_model", "optimizer", "scheduler"):
        globals().pop(name, None)

    # An interrupted Trainer/generate call can survive through traceback locals.
    for name in ("last_traceback", "last_value", "last_exc"):
        value = getattr(sys, name, None)
        tb = value if name == "last_traceback" else getattr(value, "__traceback__", None)
        if tb is not None:
            traceback.clear_frames(tb)
        if hasattr(sys, name):
            setattr(sys, name, None)
    shell = globals().get("get_ipython", lambda: None)()
    if shell is not None:
        shell.run_line_magic("reset", "-f out")  # Clear cached Python results.
        shell.last_execution_result = None
        tb = getattr(shell.InteractiveTB, "tb", None)
        if tb is not None:
            traceback.clear_frames(tb)
            shell.InteractiveTB.tb = None
    gc.collect()
    if not torch.cuda.is_initialized():
        print("CUDA is not initialized; Python GPU references cleared.")
        return
    try:
        torch.cuda.empty_cache()
        print(f"GPU cleanup: allocated {torch.cuda.memory_allocated()/1024**3:.2f} GiB; "
              f"reserved {torch.cuda.memory_reserved()/1024**3:.2f} GiB.")
    except RuntimeError as error:
        print(f"CUDA cleanup failed: {error}. Restart the kernel before loading another model.")

## 2. Original candle data — five rows per sample

Show the first, middle, and last five source rows without changing the CSV or
rounding its numeric strings. Existing ROC blanks are valid optional values.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from html import escape
from IPython.display import HTML

def show_data_samples(frame, title, raw_csv=False):
    count = 5
    middle_start = max(0, len(frame) // 2 - count // 2)
    for label, start in (("First", 0), ("Middle", middle_start), ("Last", max(0, len(frame) - count))):
        sample = frame.iloc[start:start + count]
        display(HTML(f"<h3>{escape(title)} — {label.lower()} {count} rows</h3>"
                     f"<p>Rows {start + 1:,}–{start + len(sample):,} of {len(frame):,}.</p>"))
        table = sample.style.format(escape="html").hide(axis="index")
        if raw_csv:
            table = table.set_properties(subset=["input"], **{
                "white-space": "pre-wrap", "font-family": "monospace", "text-align": "left", "min-width": "600px"})
        display(table.set_table_attributes('style="display:block;overflow-x:auto"'))

original_view = pd.read_csv(SOURCE, dtype=str, keep_default_na=False)
original_view.insert(0, "source_row", range(1, len(original_view) + 1))
show_data_samples(original_view, "Original candle data")


## 3. Generate and inspect the corrupted synthetic dataset

One CPU generation call creates 10,000 train, 1,000 validation, and 2,000 test
examples. It splits source records chronologically before constructing four-row
windows and corruption variants. Every positive target is verified offline.
If the saved dataset exists, reuse it and verify its hashes.

Show five examples from the start, middle, and end of the saved train/validation/
test concatenation. Each JSONL row contains a full corrupted CSV window. Private
reference sidecars are available to the oracle and evaluation code only.


In [ ]:
from candleprep.datasets import generate_dataset, load_jsonl

if not (DATA_DIR / "manifest.json").exists():
    generate_dataset(SOURCE, DATA_DIR, train_size=10000, validation_size=1000, test_size=2000, seed=SEED)
manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
assert hashlib.sha256(SOURCE.read_bytes()).hexdigest() == manifest["source"]["sha256"]
for filename, expected in manifest["artifact_sha256"].items():
    assert hashlib.sha256((DATA_DIR / filename).read_bytes()).hexdigest() == expected, filename
data = {}
metadata = {}
synthetic_rows = []
for split in ("train", "validation", "test"):
    data[split] = load_jsonl(DATA_DIR / (split + ".jsonl"))
    metadata[split] = load_jsonl(DATA_DIR / (split + ".references.jsonl"))
    print(split, len(data[split]), manifest["splits"][split]["classes"])
    for record in data[split]:
        synthetic_rows.append({"dataset_row": len(synthetic_rows) + 1, "split": split,
                               "id": record["id"], "input": record["input"],
                               "recoverable": record["target"]["recoverable"],
                               "operations": json.dumps(record["target"]["operations"])})
synthetic_view = pd.DataFrame(synthetic_rows)
show_data_samples(synthetic_view, "Corrupted synthetic dataset", raw_csv=True)
coverage = pd.DataFrame({split: details["categories"] for split, details in manifest["splits"].items()}).fillna(0).astype(int)
display(coverage)
coverage.plot.bar(figsize=(10, 4), title="Corruption category coverage")
plt.ylabel("Examples containing category")
plt.tight_layout()
plt.show()


## 4. Build the verified training dataset for LoRA SFT

**This section constructs the actual supervised training dataset.** Each pair is
**corrupted CSV input → oracle JSON instruction label**. The deterministic tools
execute the label offline, and the oracle checks the reconstructed whole window
against its original. The five examples below make that correspondence visible:
three positive reconstructions and two whole-window omission decisions.

The reconstructed CSV is **evidence that the label is correct**, not the text
the model learns to output. The SFT target remains the JSON tool instructions.
Originals, reconstruction evidence, and history are excluded from model features.
Negative labels remain in training and have no reconstructed window.


In [ ]:
from datasets import Dataset
from candleprep.oracle.executor import execute
from candleprep.oracle.verifier import equivalent

sft_dataset = Dataset.from_list(data["train"])
assert len(sft_dataset) == 10000
print(sft_dataset)
print("SFT feature: input; supervised target: target. id is an evaluation join key only.")

def reconstruction_preview(records, references):
    reference_by_id = {reference["id"]: reference for reference in references}
    positive = []
    for operation in ("repair_number_spelling", "normalize_numeric_format", "restore_column_layout"):
        positive.append(next(record for record in records
                             if record["target"]["recoverable"] and operation in record["target"]["operations"]
                             and record["id"] not in {selected["id"] for selected in positive}))
    negative = [record for record in records if not record["target"]["recoverable"]][:2]
    preview = []
    for record in positive + negative:
        original = reference_by_id[record["id"]]["original"]
        if record["target"]["recoverable"]:
            reconstructed = execute(record["input"], record["target"]["operations"])
            verified = equivalent(reconstructed, original)
            assert verified, record["id"]
            action = "Reconstruct complete window"
        else:
            reconstructed = "OMIT complete window; no reconstruction"
            verified = None
            action = "Omit complete window"
        preview.append({"id": record["id"], "corrupted_input": record["input"],
                        "sft_json_label": json.dumps(record["target"]),
                        "deterministic_reconstruction": reconstructed,
                        "oracle_original": original, "matches_original": verified, "action": action})
    return pd.DataFrame(preview)

verified_preview = reconstruction_preview(data["train"], metadata["train"])
display(verified_preview.style.format(escape="html").hide(axis="index").set_properties(
    subset=["corrupted_input", "sft_json_label", "deterministic_reconstruction", "oracle_original"],
    **{"white-space": "pre-wrap", "font-family": "monospace", "text-align": "left", "min-width": "320px"}
).set_table_attributes('style="display:block;overflow-x:auto"'))


## 5. Instruction-only evaluation metrics

Score recoverability, ordered operation selection, complete-answer correctness,
and JSON validity separately. Operation accuracy uses only oracle-positive
references. Invalid predictions remain failures in applicable denominators.
These functions never execute recovery tools; reconstruction above belongs to
offline label verification, and final model-directed execution is separate.


In [ ]:
from candleprep.recovery.registry import REGISTRY

KNOWN_OPERATIONS = frozenset(REGISTRY)

def strict_json(text):
    def unique_pairs(pairs):
        result = {}
        for key, value in pairs:
            if key in result:
                raise ValueError("Duplicate JSON key")
            result[key] = value
        return result
    def reject_constant(value):
        raise ValueError("Non-JSON constant: " + value)
    if not isinstance(text, str):
        return None
    try:
        value = json.loads(text, object_pairs_hook=unique_pairs, parse_constant=reject_constant)
    except (ValueError, TypeError, RecursionError):
        return None
    return value if isinstance(value, dict) else None

def score_prediction(prediction, reference):
    value = strict_json(prediction)
    decision = value.get("recoverable") if value is not None else None
    decision_valid = type(decision) is bool
    operations = value.get("operations") if value is not None else None
    operations_valid = type(operations) is list and all(type(name) is str and name in KNOWN_OPERATIONS for name in operations)
    valid = (value is not None and set(value) == {"recoverable", "operations"}
             and decision_valid and operations_valid and (decision or not operations))
    return {
        "recoverability_correct": decision_valid and decision == reference["recoverable"],
        "operation_sequence_correct": operations_valid and operations == reference["operations"],
        "full_answer_correct": bool(valid and value == reference),
        "format_valid": bool(valid),
        "predicted_class": str(decision).lower() if decision_valid else "invalid",
    }

def score_predictions(predictions, records):
    if len(predictions) != len(records) or not records:
        raise ValueError("Need one prediction per reference and a nonempty dataset")
    counts = Counter()
    confusion = {"false": {"false": 0, "true": 0, "invalid": 0},
                 "true": {"false": 0, "true": 0, "invalid": 0}}
    for prediction, record in zip(predictions, records):
        reference = record["target"]
        result = score_prediction(prediction, reference)
        counts["recoverability"] += int(result["recoverability_correct"])
        counts["full_answer"] += int(result["full_answer_correct"])
        counts["format"] += int(result["format_valid"])
        if reference["recoverable"]:
            counts["positive"] += 1
            counts["sequence"] += int(result["operation_sequence_correct"])
        confusion[str(reference["recoverable"]).lower()][result["predicted_class"]] += 1
    return {
        "examples": len(records), "oracle_positive_examples": counts["positive"],
        "recoverability_accuracy": counts["recoverability"] / len(records),
        "operation_sequence_accuracy": counts["sequence"] / counts["positive"] if counts["positive"] else None,
        "full_answer_accuracy": counts["full_answer"] / len(records),
        "format_validity": counts["format"] / len(records), "confusion_matrix": confusion,
    }

def score_breakdowns(predictions, records, references):
    if len(references) != len(records) or any(a["id"] != b["id"] for a, b in zip(records, references)):
        raise ValueError("Evaluation metadata must match record IDs")
    groups = {}
    for index, reference in enumerate(references):
        names = list(reference["categories"])
        names.append("single" if reference["composition_size"] == 1 else "composed")
        for name in names:
            groups.setdefault(name, []).append(index)
    output = {}
    for name, indices in groups.items():
        output[name] = score_predictions([predictions[i] for i in indices], [records[i] for i in indices])
    return output


## 6. Fixed model prompt and target-only token loss

Model inputs contain only the static task/schema and corrupted CSV. Prompt and
padding tokens are masked with -100. Preserve the full JSON target and reject
overlong samples instead of truncating their answers. Decode only newly generated
tokens. The same prompt and generation settings apply before and after SFT.


In [ ]:
TASK = """Given a corrupted candle CSV window, predict the canonical shortest repair plan under the supported tools.
Return only JSON with exactly two keys: recoverable (a Boolean) and operations (an ordered list of tool names).
Use {"recoverable":false,"operations":[]} when exact recovery of the complete window is unsupported. The complete window is omitted in that case.
Schema: time,open,high,low,close,Volume,ROC. All fields are required except ROC, which may be blank. Negative ROC is valid.
Timestamps require an explicit UTC offset. Irregular market intervals are allowed. Preserve all rows and exact numeric meaning.
Supported conventions: English cardinal words and individual fractional digits; minus for negatives; point or normalized decimal;
USD currency words; currency prefix $; suffixes USD/contracts/percent; decimal comma without thousands grouping;
cents explicitly identify a scale of 100; declared removable noise is ? @ ~; doubled decimal punctuation;
alternate date DD/MM/YYYY HH:MM:SS followed by numeric UTC offset; column headers move with column values;
cyclic row rotations identifiable by time; exact timestamp-identical duplicates; chronological row sorting.
Arbitrary digit/sign changes, lost decimals/digits, missing required values, conflicting duplicates, and missing rows are unsupported.
Operation names: """ + ", ".join(sorted(KNOWN_OPERATIONS)) + """.
For equally short valid plans, choose the lexicographically first ordered operation list.
Do not include unnecessary validator calls. Valid data needing no transforms uses recoverable true and an empty operations list.
Corrupted CSV:
"""

def user_text(record):
    return TASK + record["input"]

def prompt_tokens(tokenizer, record):
    return tokenizer.apply_chat_template([{"role": "user", "content": user_text(record)}], tokenize=True, add_generation_prompt=True, return_dict=False)

def tokenize_record(record, tokenizer):
    messages = [{"role": "user", "content": user_text(record)}]
    prompt = prompt_tokens(tokenizer, record)
    target = json.dumps(record["target"], separators=(",", ":"))
    full = tokenizer.apply_chat_template(messages + [{"role": "assistant", "content": target}], tokenize=True, return_dict=False)
    if full[:len(prompt)] != prompt:
        raise ValueError("Chat template does not preserve the prompt boundary")
    if len(full) > MAX_LENGTH:
        raise ValueError(f"Sample {record['id']} needs {len(full)} tokens; increase MAX_LENGTH without dropping its target")
    return {"input_ids": full, "attention_mask": [1] * len(full),
            "labels": [-100] * len(prompt) + full[len(prompt):]}


## 7. Mistral/Transformers/PEFT helpers

Retain the starter's Mistral-7B, 4-bit NF4/double quantization, and PEFT-LoRA
framework. Each rank starts from a fresh base model. These definitions do not
load the model; the before-SFT section below does. Dataset IDs, labels, and oracle
metadata are never prompt features.


In [ ]:
import time
import torch
from tqdm.auto import tqdm
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          DataCollatorForSeq2Seq, Trainer, TrainerCallback, TrainingArguments, set_seed,
                          StoppingCriteria, StoppingCriteriaList)
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

def duration(seconds):
    if seconds is None:
        return "estimating"
    hours, remainder = divmod(max(0, int(seconds)), 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

def report_phase(message):
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)

class TokenProgress(StoppingCriteria):
    """Observe generated-token counts without changing when generation stops."""
    def __init__(self, bar, prompt_length, example_number):
        self.bar = bar
        self.prompt_length = prompt_length
        self.example_number = example_number
        self.last_update = time.monotonic() - 5

    def __call__(self, input_ids, scores, **kwargs):
        now = time.monotonic()
        if now - self.last_update >= 5:
            generated = input_ids.shape[-1] - self.prompt_length
            self.bar.set_postfix_str(
                f"example {self.example_number}: {generated}/{MAX_NEW_TOKENS} new tokens", refresh=True)
            self.last_update = now
        return False

def load_tokenizer():
    report_phase("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer

def load_base():
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required for the model stages")
    free, _ = torch.cuda.mem_get_info()
    minimum_free = globals().get("MIN_FREE_VRAM_GIB", 7.5)
    if free < minimum_free * 1024**3:
        raise RuntimeError(f"Only {free/1024**3:.2f} GiB GPU memory is free; "
                           f"the configured availability guard is {minimum_free:.1f} GiB. "
                           "Release other GPU workloads before continuing.")
    set_seed(SEED)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
    report_phase(f"Loading quantized base model: {MODEL_ID}; free GPU {free/1024**3:.2f} GiB.")
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=config,
                                               device_map={"": 0}, attn_implementation="sdpa")
    report_phase("Base model loaded.")
    return model

def save_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")

def save_records(path, records):
    with Path(path).open("w", encoding="utf-8") as stream:
        for record in records:
            stream.write(json.dumps(record) + "\n")

def predict(model, tokenizer, records, phase="Generating responses"):
    model.eval()
    model.config.use_cache = True
    predictions = []
    started = time.monotonic()
    last_report = started
    report_phase(f"{phase}: starting {len(records):,} examples; this is inference, not training.")
    with tqdm(total=len(records), desc=phase, unit="example", mininterval=1,
              dynamic_ncols=True) as bar:
        for index, record in enumerate(records):
            ids = prompt_tokens(tokenizer, record)
            if len(ids) + MAX_NEW_TOKENS > MAX_LENGTH:
                raise ValueError("Prompt exceeds the frozen generation budget")
            bar.set_postfix_str(f"example {index + 1}: preparing {len(ids)} prompt tokens", refresh=True)
            inputs = torch.tensor([ids], device=model.get_input_embeddings().weight.device)
            observer = TokenProgress(bar, len(ids), index + 1)
            with torch.inference_mode():
                generated = model.generate(input_ids=inputs, attention_mask=torch.ones_like(inputs),
                                           max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                           pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
                                           stopping_criteria=StoppingCriteriaList([observer]))
            predictions.append(tokenizer.decode(generated[0, inputs.shape[1]:], skip_special_tokens=True).strip())
            bar.update(1)
            now = time.monotonic()
            if now - last_report >= 30 or index + 1 == len(records):
                elapsed = now - started
                completed = index + 1
                rate = completed / max(elapsed, 1e-9)
                remaining = (len(records) - completed) / rate
                report_phase(f"{phase}: {completed:,}/{len(records):,} complete | elapsed {duration(elapsed)} "
                             f"| {elapsed/completed:.1f} sec/example | ETA {duration(remaining)}")
                last_report = now
    report_phase(f"{phase}: finished in {duration(time.monotonic() - started)}.")
    return predictions

def evaluate_and_save(model, tokenizer, split, label):
    phase = f"{label} / {split}"
    predictions = predict(model, tokenizer, data[split], phase=phase)
    report_phase(f"{phase}: scoring predictions and saving artifacts...")
    report = score_predictions(predictions, data[split])
    report["breakdowns"] = score_breakdowns(predictions, data[split], metadata[split])
    save_records(RESULTS / (label + "_" + split + "_predictions.jsonl"), [
        {"id": record["id"], "prediction": prediction, "reference": record["target"]}
        for record, prediction in zip(data[split], predictions)])
    save_json(RESULTS / (label + "_" + split + "_metrics.json"), report)
    report_phase(f"{phase}: metrics and predictions saved.")
    return report

def run_environment():
    from importlib.metadata import version
    from candleprep import __version__
    return {"platform": platform.platform(), "python": platform.python_version(),
            "gpu": torch.cuda.get_device_name(), "model": MODEL_ID, "seed": SEED,
            "candleprep": __version__, "max_length": MAX_LENGTH, "max_new_tokens": MAX_NEW_TOKENS,
            "prompt_sha256": hashlib.sha256(TASK.encode()).hexdigest(),
            "dataset_artifacts": manifest["artifact_sha256"],
            "packages": {name: version(name) for name in ("torch", "transformers", "datasets", "peft", "bitsandbytes")}}

def release_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 8. Tokenize the complete SFT training set

Build the model-ready dataset from every verified training pair. The original
`id`, `input`, and `target` fields are replaced by token IDs, attention masks,
and target-only loss labels. No original/reconstructed CSV becomes a feature.
Tokenization is done once and reused for every LoRA rank.


In [ ]:
tokenizer = load_tokenizer()
tokenized_train = Dataset.from_list([tokenize_record(record, tokenizer) for record in sft_dataset])
assert len(tokenized_train) == len(sft_dataset) == 10000
assert set(tokenized_train.column_names) == {"input_ids", "attention_mask", "labels"}
print(tokenized_train)
print("Longest complete training sequence:", max(len(row["input_ids"]) for row in tokenized_train))
print("Optimizer steps per full epoch:", math.ceil(len(tokenized_train) / (BATCH_SIZE * GRADIENT_ACCUMULATION)))


## 9. Before SFT — exactly three model examples

Load the original quantized Mistral model and generate only the three fixed
validation examples. Freeze their IDs, exact prompts, targets, and generation
settings. Display targets separately; never send them to the model.

`candleprep` already generated and verified all training labels offline. This
demonstration does not construct labels or run the full validation/test sets.
The following SFT section consumes saved input/target pairs directly.


In [ ]:
def choose_demo_records(records, references):
    if len(records) != len(references):
        raise ValueError("Demo metadata must match the record count")
    joined = list(zip(records, references))
    if any(record["id"] != reference["id"] for record, reference in joined):
        raise ValueError("Demo metadata must match record IDs")
    lexical = next(record for record, reference in joined
                   if record["target"]["recoverable"] and "repair_number_spelling" in record["target"]["operations"])
    composed = next(record for record, reference in joined
                    if record["target"]["recoverable"] and len(reference["categories"]) > 1
                    and len(record["target"]["operations"]) > 1 and record["id"] != lexical["id"])
    negative = next(record for record, reference in joined
                    if not record["target"]["recoverable"]
                    and any(event["operator"] == "remove_required_column" for event in reference["history"]))
    return [lexical, composed, negative]

def choose_inspection_records(records, matched_records):
    """Keep the three frozen comparisons and add two distinct repair cases."""
    chosen = list(matched_records)
    used = {record["id"] for record in chosen}
    for operation in ("normalize_numeric_format", "deduplicate_records"):
        extra = next(record for record in records if record["id"] not in used
                     and record["target"]["recoverable"]
                     and operation in record["target"]["operations"])
        chosen.append(extra)
        used.add(extra["id"])
    return chosen

def render_prompt(record):
    return tokenizer.decode(prompt_tokens(tokenizer, record), skip_special_tokens=False)

def show_model_examples(records, predictions, before=None):
    if len(records) != len(predictions):
        raise ValueError("Each example requires exactly one prediction")
    before_by_id = {example["id"]: example for example in (before or [])}
    rows = []
    for index, (record, prediction) in enumerate(zip(records, predictions)):
        row = {"example": index + 1, "id": record["id"], "model_received_input": render_prompt(record)}
        if before is not None:
            previous = before_by_id.get(record["id"])
            row["before_sft_output"] = (previous["prediction"] if previous else
                                         "Not collected: additional after-SFT inspection case")
        row["model_output" if before is None else "after_sft_output"] = prediction
        row["expected_json"] = json.dumps(record["target"])
        row["full_answer_correct"] = score_prediction(prediction, record["target"])["full_answer_correct"]
        rows.append(row)
    frame = pd.DataFrame(rows)
    display(frame.style.format(escape="html").hide(axis="index").set_properties(
        **{"white-space": "pre-wrap", "font-family": "monospace", "text-align": "left", "min-width": "220px"}
    ).set_table_attributes('style="display:block;overflow-x:auto"'))

demo_records = choose_demo_records(data["validation"], metadata["validation"])
print("Frozen validation example IDs:", [record["id"] for record in demo_records])


### Generate and save the three before-SFT responses

This cell makes three demonstration predictions, saves them, and releases the
base model. Continue to SFT immediately afterward; no full-set baseline follows.


In [ ]:
report_phase("BEFORE SFT: generate exactly three demonstration responses; no full-set inference.")
assert len(demo_records) == 3
base_model = load_base()
try:
    before_predictions = predict(base_model, tokenizer, demo_records, phase="Before SFT: three fixed examples")
    show_model_examples(demo_records, before_predictions)
    before_examples = [{"id": record["id"], "prompt": render_prompt(record),
                        "prediction": prediction, "reference": record["target"]}
                       for record, prediction in zip(demo_records, before_predictions)]
    save_records(RESULTS / "demo_before.jsonl", before_examples)
    save_json(RESULTS / "demo_generation.json", {"model": MODEL_ID, "seed": SEED,
              "max_new_tokens": MAX_NEW_TOKENS, "do_sample": False})
    save_json(RESULTS / "demo_before_metrics.json", score_predictions(before_predictions, demo_records))
    save_json(RESULTS / "environment.json", run_environment())
finally:
    del base_model
    release_gpu()
report_phase("Three-example baseline finished. Continue directly to SFT using the saved input/label pairs.")


## 10. Full LoRA SFT using the offline labels

Follow the starter's direct SFT path: tokenized corrupted input + verified JSON
target → Trainer forward pass → target-token cross-entropy loss → LoRA updates.
Training does not call `model.generate()` or execute recovery tools. Target
tokens are provided during training (teacher forcing); prompt/padding loss is masked.

Train the configured `LORA_RANK` once on all 10,000 examples for one epoch.
Batch size 1 with accumulation 16 gives 625 optimizer steps. Save checkpoints
every 100 steps, retain two, and resume matching interrupted runs. Completed
adapters are reused; choose another RESULTS directory for changed settings.

Progress shows optimizer steps, accumulation activity, elapsed time, ETA, GPU
allocation, first/every-five-step loss, and checkpoint completion. Optional rank
comparisons are deferred to the evaluation section after the main SFT workflow.


In [ ]:
class TrainingFeedback(TrainerCallback):
    """Read-only feedback between accumulation microbatches and optimizer steps."""
    def __init__(self, rank):
        self.rank = rank

    def on_train_begin(self, args, state, control, **kwargs):
        self.started = time.monotonic()
        self.last_report = self.started - 15
        self.initial_step = state.global_step
        self.microbatches = 0
        report_phase(f"LoRA rank {self.rank}: training begins at optimizer step "
                     f"{state.global_step}/{state.max_steps}. One step accumulates "
                     f"{args.gradient_accumulation_steps} microbatches.")

    def feedback(self, args, state, force=False):
        now = time.monotonic()
        if not force and now - self.last_report < 15:
            return
        elapsed = now - self.started
        completed = state.global_step - self.initial_step
        remaining = ((state.max_steps - state.global_step) * elapsed / completed
                     if completed else None)
        batch = (f" | accumulating {self.microbatches}/{args.gradient_accumulation_steps}"
                 if self.microbatches else "")
        gpu = (f" | allocated GPU {torch.cuda.memory_allocated()/1024**3:.2f} GiB"
               if torch.cuda.is_available() else "")
        report_phase(f"LoRA rank {self.rank}: optimizer step {state.global_step}/{state.max_steps}"
                     f"{batch} | elapsed {duration(elapsed)} | ETA {duration(remaining)}{gpu}")
        self.last_report = now

    def on_substep_end(self, args, state, control, **kwargs):
        self.microbatches += 1
        self.feedback(args, state)

    def on_step_end(self, args, state, control, **kwargs):
        self.microbatches = 0
        self.feedback(args, state, force=(state.global_step == self.initial_step + 1))

    def on_save(self, args, state, control, **kwargs):
        report_phase(f"LoRA rank {self.rank}: checkpoint saved at optimizer step {state.global_step}.")

    def on_train_end(self, args, state, control, **kwargs):
        report_phase(f"LoRA rank {self.rank}: training finished at step {state.global_step} "
                     f"after {duration(time.monotonic() - self.started)}.")

def train_adapter(rank, destination):
    configuration = {"rank": rank, "model": MODEL_ID, "epochs": NUM_EPOCHS,
                     "examples": len(tokenized_train), "batch_size": BATCH_SIZE,
                     "gradient_accumulation": GRADIENT_ACCUMULATION,
                     "learning_rate": LEARNING_RATE, "seed": SEED, "max_length": MAX_LENGTH,
                     "lora_alpha": 32, "prompt_sha256": hashlib.sha256(TASK.encode()).hexdigest(),
                     "dataset_artifacts": manifest["artifact_sha256"]}
    config_path = destination / "configuration.json"

    if config_path.exists():
        if json.loads(config_path.read_text()) != configuration:
            raise ValueError("Existing run configuration differs; choose a new RESULTS directory")
        if (destination / "training.json").exists() and (destination / "adapter/adapter_model.safetensors").exists():
            print("Reusing completed full-data adapter:", destination)
            return destination / "adapter"
        
    elif destination.exists() and any(destination.iterdir()):
        raise ValueError("Run directory is nonempty without matching configuration")
    
    destination.mkdir(parents=True, exist_ok=True)
    save_json(config_path, configuration)
    checkpoint = get_last_checkpoint(str(destination))
    report_phase(f"LoRA rank {rank}: preparing model and adapters; resume checkpoint: {checkpoint or 'none'}.")
    model = prepare_model_for_kbit_training(load_base())

    model = get_peft_model(model, LoraConfig(
        r=rank, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    
    model.config.use_cache = False
    model.print_trainable_parameters()

    arguments = TrainingArguments(
        output_dir=str(destination), per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION, num_train_epochs=NUM_EPOCHS,
        max_steps=-1, learning_rate=LEARNING_RATE, logging_steps=5,
        logging_first_step=True, disable_tqdm=False,
        optim="paged_adamw_8bit", bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(), gradient_checkpointing=True,
        save_strategy="steps", save_steps=100, save_total_limit=2,
        report_to="none", seed=SEED, data_seed=SEED, remove_unused_columns=False)
    
    trainer = Trainer(model=model, args=arguments, train_dataset=tokenized_train,
                      callbacks=[TrainingFeedback(rank)],
                      processing_class=tokenizer,
                      data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, label_pad_token_id=-100))
    
    torch.cuda.reset_peak_memory_stats()

    try:
        report_phase(f"LoRA rank {rank}: full training on {len(tokenized_train):,} examples for {NUM_EPOCHS} epoch(s).")
        result = trainer.train(resume_from_checkpoint=checkpoint)
        if not math.isfinite(result.training_loss):
            raise RuntimeError("Training loss is not finite")
        report_phase(f"LoRA rank {rank}: saving completed adapter...")
        adapter_dir = destination / "adapter"
        trainer.model.save_pretrained(adapter_dir)
        tokenizer.save_pretrained(adapter_dir)
        trainer.state.save_to_json(str(destination / "trainer_state.json"))
        save_json(destination / "training.json", {"rank": rank, "training_examples": len(tokenized_train),
                  "configured_epochs": NUM_EPOCHS, "completed_epochs": trainer.state.epoch,
                  "optimizer_steps": trainer.state.global_step, "training_loss": result.training_loss,
                  "peak_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3,
                  "peak_reserved_gib": torch.cuda.max_memory_reserved() / 1024**3,
                  "environment": run_environment()})
        return adapter_dir
    
    finally:
        del trainer, model
        release_gpu()

report_phase("Training function ready. This definition cell does not train; execute the following full-training cell to start.")


### Execute the full SFT run

This cell starts Trainer on the saved, tokenized labeled dataset. It performs
no model-response generation before, during, or after fitting the adapter.


In [ ]:
if not (RESULTS / "demo_before.jsonl").exists():
    raise RuntimeError("Run the three-example before-SFT demonstration first")

report_phase(f"START SFT: rank {LORA_RANK}, {len(tokenized_train):,} saved labeled examples. No response generation is used to train.")
selected_adapter = train_adapter(LORA_RANK, RESULTS / ("rank_" + str(LORA_RANK)))
save_json(RESULTS / "selection.json", {"rank": LORA_RANK, "selected_using": "configured primary SFT rank"})
report_phase("SFT finished. The next section generates five after-SFT responses (three matched, two additional).")


## 11. After SFT — five inspection examples

Load the configured trained adapter onto a fresh base. Keep the three frozen
IDs, prompts, and generation settings from Section 9 for matched before/after
comparisons. Add two more examples: numeric formatting and duplicate removal.
All five show the received input, actual output, and expected JSON. The extra
two are marked as after-only inspections because no baseline was collected.

Save paired metrics for the original three separately from five-case inspection
metrics. Neither small demonstration estimates full held-out accuracy. Rerun
this cell to inspect five results using the saved adapter; no retraining is needed.

### Resuming after a kernel restart

The saved adapter survives a restart; Python variables and function definitions
do not. Before running this section in a fresh kernel:

1. Run Sections 1–8 to restore imports, cached datasets, helper definitions,
   tokenizer, and tokenized training data. These setup cells do not train a model.
2. Run the first helper-definition cell in Section 9 (`choose_demo_records`,
   `render_prompt`, etc.). Skip its before-SFT generation cell.
3. To run the optional rank study, also run the function-definition cell in
   Section 10 (`TrainingFeedback` and `train_adapter`). Skip the SFT execution cell.
4. Run Section 11 below to reload the saved adapter and inspect five answers.
   Run the desired Section 12 jobs separately afterward.

Old displayed outputs do not mean those definitions exist in the current kernel.
GPU cleanup preserves setup variables; restarting the kernel removes them.


In [ ]:
_missing_setup = [name for name in ('report_phase', 'load_jsonl', 'data', 'tokenizer', 'render_prompt', 'choose_inspection_records', 'PeftModel') if name not in globals()]
if _missing_setup:
    raise RuntimeError("Kernel setup is incomplete (missing: " + ", ".join(_missing_setup) +
                       "). Follow the restart checklist at the start of Section 11; "
                       "rerun setup/definition cells, then this cell. Saved adapters do not need retraining.")

report_phase("AFTER SFT: three matched comparisons plus two additional inspection cases.")

before_examples = load_jsonl(RESULTS / "demo_before.jsonl")

assert json.loads((RESULTS / "demo_generation.json").read_text()) == {
    "model": MODEL_ID, "seed": SEED, "max_new_tokens": MAX_NEW_TOKENS, "do_sample": False
}, "Before/after generation settings changed"

validation_by_id = {record["id"]: record for record in data["validation"]}
matched_records = [validation_by_id[example["id"]] for example in before_examples]

assert len(matched_records) == 3

for record, example in zip(matched_records, before_examples):
    assert render_prompt(record) == example["prompt"], "Before/after prompts changed"

inspection_records = choose_inspection_records(data["validation"], matched_records)

assert len(inspection_records) == 5

selection = json.loads((RESULTS / "selection.json").read_text())

selected_adapter = RESULTS / ("rank_" + str(selection["rank"])) / "adapter"

model = PeftModel.from_pretrained(load_base(), selected_adapter)

try:
    after_predictions = predict(model, tokenizer, inspection_records, phase="After SFT: five inspection examples")
    show_model_examples(inspection_records, after_predictions, before_examples)
    save_records(RESULTS / "demo_after.jsonl", [{"id": record["id"], "prompt": render_prompt(record),
                 "prediction": prediction, "reference": record["target"]}
                 for record, prediction in zip(inspection_records, after_predictions)])
    # Preserve the same three-case denominator for the before/after comparison.
    paired_metrics = score_predictions(after_predictions[:3], matched_records)
    inspection_metrics = score_predictions(after_predictions, inspection_records)
    save_json(RESULTS / "demo_after_metrics.json", paired_metrics)
    save_json(RESULTS / "inspection_after_metrics.json", inspection_metrics)
    print("Matched before/after cases (3):", paired_metrics)
    print("All after-SFT inspection cases (5):", inspection_metrics)

finally:
    del model
    release_gpu()


## 12. Separate evaluation — optional to execute now, required for final reporting

The main path above is now complete: three before examples → full SFT from saved
labels → five after examples. These extra evaluation jobs do not create or
reconstruct the training dataset, and do not block that path.

Enable `RUN_RANK_STUDY` when ready to compare 8/16/32 on validation data. It reuses
any completed matching adapter. The study reports differences without silently
changing the configured primary adapter; choose settings using validation only.

Enable `RUN_FULL_TEST_COMPARISON` when ready for the assignment's full quantitative
comparison. A fresh, unadapted base and the configured trained adapter each
generate responses to the same 2,000 held-out inputs. The base weights are still
the original weights even though this measurement occurs later in notebook order.
This is intentionally a long, explicit job. Four instruction metrics, confusion
matrices, and category/composition breakdowns follow from real predictions.

**Full evaluation findings:** Pending these explicitly enabled evaluation jobs.


In [ ]:
_missing_setup = [name for name in ('report_phase', 'RUN_RANK_STUDY') if name not in globals()]
if _missing_setup:
    raise RuntimeError("Kernel setup is incomplete (missing: " + ", ".join(_missing_setup) +
                       "). Follow the restart checklist at the start of Section 11; "
                       "rerun setup/definition cells, then this cell. Saved adapters do not need retraining.")

report_phase(f"Kernel setting: RUN_RANK_STUDY={RUN_RANK_STUDY}")

if RUN_RANK_STUDY:
    rank_reports = []
    for rank in RANKS:
        report_phase(f"OPTIONAL RANK STUDY: rank {rank}; train/reuse adapter, then validation inference.")
        adapter = train_adapter(rank, RESULTS / ("rank_" + str(rank)))
        model = PeftModel.from_pretrained(load_base(), adapter)
        try:
            report = evaluate_and_save(model, tokenizer, "validation", "rank_" + str(rank))
            rank_reports.append({"rank": rank, **report})
        finally:
            del model
            release_gpu()
    save_json(RESULTS / "rank_comparison.json", rank_reports)
    display(pd.DataFrame(rank_reports).drop(columns=["breakdowns", "confusion_matrix"]))
    
else:
    report_phase("Optional rank study disabled; no training or validation generation started here.")
    report_phase("After editing a switch, rerun the configuration cell, then this cell. Editing text alone does not update the kernel.")


### Explicit full test comparison

Run this later when ready for 2,000 original-base and 2,000 trained-adapter test
generations. It evaluates two model conditions; the training inputs and oracle
labels are reused unchanged. The small demonstrations do not require it.


In [ ]:
_missing_setup = [name for name in ('report_phase', 'RUN_FULL_TEST_COMPARISON') if name not in globals()]
if _missing_setup:
    raise RuntimeError("Kernel setup is incomplete (missing: " + ", ".join(_missing_setup) +
                       "). Follow the restart checklist at the start of Section 11; "
                       "rerun setup/definition cells, then this cell. Saved adapters do not need retraining.")

report_phase(f"Kernel setting: RUN_FULL_TEST_COMPARISON={RUN_FULL_TEST_COMPARISON}")

if RUN_FULL_TEST_COMPARISON:
    assert json.loads((RESULTS / "demo_generation.json").read_text()) == {
        "model": MODEL_ID, "seed": SEED, "max_new_tokens": MAX_NEW_TOKENS, "do_sample": False
    }, "Evaluation generation settings changed from the demonstrations"
    report_phase("FULL TEST COMPARISON: original base, then trained adapter, each on the same 2,000 examples.")
    model = load_base()
    try:
        display(evaluate_and_save(model, tokenizer, "test", "base"))
    finally:
        del model
        release_gpu()
    selection = json.loads((RESULTS / "selection.json").read_text())
    selected_adapter = RESULTS / ("rank_" + str(selection["rank"])) / "adapter"
    model = PeftModel.from_pretrained(load_base(), selected_adapter)
    try:
        display(evaluate_and_save(model, tokenizer, "test", "selected"))
    finally:
        del model
        release_gpu()
        
else:
    report_phase("Full test generation disabled. The cached-label SFT and three-example comparisons do not depend on it.")
    report_phase("After editing a switch, rerun the configuration cell, then this cell. Editing text alone does not update the kernel.")


### Display measured evaluation results

Render tables and charts from actual saved predictions. No model is loaded and
no response generation occurs in this display cell.


In [ ]:
metric_names = ["recoverability_accuracy", "operation_sequence_accuracy", "full_answer_accuracy", "format_validity"]
test_reports = {}
for name in ("base", "selected"):
    metric_path = RESULTS / (name + "_test_metrics.json")
    if metric_path.exists():
        test_reports[name] = json.loads(metric_path.read_text())
if len(test_reports) == 2:
    table = pd.DataFrame({name: {key: report[key] for key in metric_names} for name, report in test_reports.items()})
    display(table)
    table.plot.bar(ylim=(0, 1), figsize=(10, 4), title="Held-out performance before and after LoRA SFT")
    plt.tight_layout()
    plt.show()
    display(pd.DataFrame(test_reports["selected"]["confusion_matrix"]).T.rename_axis("Oracle class"))
    display(pd.DataFrame(test_reports["selected"]["breakdowns"]).T[metric_names])
else:
    print("Full before/after test metrics populate after the model stages complete.")
rank_path = RESULTS / "rank_comparison.json"
if rank_path.exists():
    rank_table = pd.DataFrame(json.loads(rank_path.read_text())).set_index("rank")[metric_names]
    display(rank_table)
    rank_table.plot.bar(ylim=(0, 1), title="Validation performance by LoRA rank", figsize=(10, 4))
    plt.tight_layout()
    plt.show()


## 13. Separate final model-directed repair demonstration

This explicitly enabled section executes saved model-predicted operations.
Negative decisions omit entire windows. Recovery tools receive no answer key;
references are consulted afterward to measure reconstruction. Keep this section
outside routine SFT, validation, test scoring, and rank comparisons.


In [ ]:
def final_execution_demo(records, references, saved_predictions, destination):
    from candleprep.oracle.executor import execute
    from candleprep.oracle.verifier import equivalent
    from candleprep.schema import RecoveryError
    if destination.exists():
        raise FileExistsError("Demonstration output already exists")
    if not (len(records) == len(references) == len(saved_predictions)):
        raise ValueError("Demonstration records must align")
    counts = Counter()
    repaired = []
    omissions = []
    outcomes = []
    for record, reference, saved in zip(records, references, saved_predictions):
        if not (record["id"] == reference["id"] == saved["id"]):
            raise ValueError("Demonstration IDs do not match")
        expected_positive = record["target"]["recoverable"]
        counts["oracle_recoverable"] += int(expected_positive)
        result = score_prediction(saved["prediction"], record["target"])
        plan = strict_json(saved["prediction"])
        status = "invalid_plan"
        reconstructed = False
        if result["format_valid"]:
            if not plan["recoverable"]:
                status = "omitted"
                omissions.append(record["id"])
                counts["false_omissions"] += int(expected_positive)
            else:
                counts["attempted_repairs_of_oracle_negative"] += int(not expected_positive)
                try:
                    restored = execute(record["input"], plan["operations"])
                    reconstructed = equivalent(restored, reference["original"])
                    status = "reconstructed" if reconstructed else "incorrect_reconstruction"
                    repaired.append({"id": record["id"], "csv": restored})
                    counts["retained"] += 1
                    counts["exact_on_oracle_recoverable"] += int(reconstructed and expected_positive)
                except RecoveryError:
                    status = "execution_failure"
        counts[status] += 1
        outcomes.append({"id": record["id"], "status": status, "exact_reconstruction": reconstructed})
    summary = dict(counts)
    summary["examples"] = len(records)
    summary["retained_coverage"] = counts["retained"] / len(records)
    summary["exact_reconstruction_on_oracle_recoverable"] = (counts["exact_on_oracle_recoverable"] / counts["oracle_recoverable"] if counts["oracle_recoverable"] else None)
    destination.mkdir(parents=True)
    save_records(destination / "repaired_windows.jsonl", repaired)
    save_json(destination / "omitted_example_ids.json", omissions)
    save_records(destination / "outcomes.jsonl", outcomes)
    save_json(destination / "summary.json", summary)
    return summary

if RUN_FINAL_DEMO:
    saved = load_jsonl(RESULTS / "selected_test_predictions.jsonl")
    display(final_execution_demo(data["test"], metadata["test"], saved, RESULTS / "final_execution"))
else:
    print("Final model-directed execution is disabled.")


## Discussion and sources

- The reconstruction preview verifies oracle labels. The SFT targets are JSON
  instructions, not repaired CSV, numeric replacements, or oracle reasoning.
- Recoverability is bounded by the supported deterministic tools and recipes.
  Plausible numeric errors may not be identifiable from the observed input alone.
- Five after-SFT examples, including three matched baseline cases, illustrate behavior; full held-out metrics support efficacy
  claims. Synthetic corruption does not establish performance on natural errors.
- The CPU library is published as
  [candleprep 0.1.0](https://pypi.org/project/candleprep/0.1.0/). Generated datasets,
  checkpoints, and local logs stay outside Git and the package distribution.
- Full A30 execution and final PDF exports remain submission steps. Preserve
  the original handout before producing the notebook export `proj5.pdf`.

Sources: approved user design; source candle CSV; supplied assignment/starter;
[PEFT quantization](https://huggingface.co/docs/peft/developer_guides/quantization),
[Transformers Trainer](https://huggingface.co/docs/transformers/main_classes/trainer),
[chat templates](https://huggingface.co/docs/transformers/chat_templating).
AI/source assistance is summarized in the separate local `proj5doc.md`.
